In [1]:
from pyspark.sql import SparkSession

In [4]:
spark=SparkSession.builder.appName('Kenya Ecommerce Transactions Analysis').getOrCreate()

In [6]:
dataset=spark.read.csv('/content/kenya_ecommerce_transactions.csv',header=True,inferSchema=True)

In [8]:
dataset.show()

+--------------------+--------------------+--------------------+---------------+-------+--------+-------------------+----------------+-------+-------+-----------+--------+-----------+-----------------+
|      transaction_id|             user_id|          product_id|       category|  price|quantity|   transaction_date|  payment_method|country|   city|device_type|user_age|user_gender|is_first_purchase|
+--------------------+--------------------+--------------------+---------------+-------+--------+-------------------+----------------+-------+-------+-----------+--------+-----------+-----------------+
|e3e70682-c209-4ca...|e47d57b2-ac92-421...|29a445ec-dccc-465...|          Books|2789.19|       2|2025-02-15 06:23:07|   Bank Transfer|  Kenya|Mombasa|     Mobile|      59|      Other|             true|
|f728b4fa-4248-4e3...|29bace2a-e4bc-459...|2521b414-4a1d-439...|          Books|3604.43|       4|2025-04-08 04:11:55|          M-Pesa|  Kenya|Garissa|     Mobile|      54|     Female|         

In [14]:
dataset.describe(['price','quantity','user_age']).show()

+-------+------------------+------------------+------------------+
|summary|             price|          quantity|          user_age|
+-------+------------------+------------------+------------------+
|  count|            100000|            100000|            100000|
|   mean|  2547.55015480001|           2.50417|          41.01701|
| stddev|1416.2844395383759|1.1176292328187447|13.560297912534725|
|    min|            100.02|                 1|                18|
|    max|           4999.89|                 4|                64|
+-------+------------------+------------------+------------------+



In [11]:
dataset.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- transaction_date: timestamp (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- user_age: integer (nullable = true)
 |-- user_gender: string (nullable = true)
 |-- is_first_purchase: boolean (nullable = true)



In [17]:
#how many trnsactons were completed?
dataset.describe().show()
#from the count we see that transaction id=10K.ie,all done

+-------+--------------------+--------------------+--------------------+---------------+------------------+------------------+--------------+-------+-------+-----------+------------------+-----------+
|summary|      transaction_id|             user_id|          product_id|       category|             price|          quantity|payment_method|country|   city|device_type|          user_age|user_gender|
+-------+--------------------+--------------------+--------------------+---------------+------------------+------------------+--------------+-------+-------+-----------+------------------+-----------+
|  count|              100000|              100000|              100000|         100000|            100000|            100000|        100000| 100000| 100000|     100000|            100000|     100000|
|   mean|                NULL|                NULL|                NULL|           NULL|  2547.55015480001|           2.50417|          NULL|   NULL|   NULL|       NULL|          41.01701|       N

In [20]:
#what was the range in commodity pirchased?
dataset.describe().show()
#range=max-Min=(4-1=3)..from quantity

+-------+--------------------+--------------------+--------------------+---------------+------------------+------------------+--------------+-------+-------+-----------+------------------+-----------+
|summary|      transaction_id|             user_id|          product_id|       category|             price|          quantity|payment_method|country|   city|device_type|          user_age|user_gender|
+-------+--------------------+--------------------+--------------------+---------------+------------------+------------------+--------------+-------+-------+-----------+------------------+-----------+
|  count|              100000|              100000|              100000|         100000|            100000|            100000|        100000| 100000| 100000|     100000|            100000|     100000|
|   mean|                NULL|                NULL|                NULL|           NULL|  2547.55015480001|           2.50417|          NULL|   NULL|   NULL|       NULL|          41.01701|       N

In [21]:
#what was the age of the oldest client?
#from user_age=64 and youngest =18
#avg price of commodities?=2547.28..mean

In [23]:
#avg price
from pyspark.sql.functions import avg
dataset.select(avg('price')).show()

+----------------+
|      avg(price)|
+----------------+
|2547.55015480001|
+----------------+



In [31]:
#how many transactions were made using mpesa
dataset.select('payment_method').distinct().show()
#we use distinct to show unique values

+----------------+
|  payment_method|
+----------------+
|     Credit Card|
|          M-Pesa|
|   Bank Transfer|
|Cash on Delivery|
+----------------+



In [34]:
#how many product categories were there
dataset.select('category').distinct().show()

+---------------+
|       category|
+---------------+
|        Fashion|
|      Groceries|
|    Electronics|
|          Books|
|Health & Beauty|
+---------------+



In [35]:
#how many cities did we have orders from
dataset.select('city').distinct().show()

+-------+
|   city|
+-------+
| Nakuru|
| Kisumu|
|Mombasa|
|Eldoret|
|Garissa|
|Nairobi|
|  Thika|
| Kitale|
|   Meru|
|  Nyeri|
+-------+



In [38]:
#how many transactions were made using mpesa
dataset.groupBy('payment_method').count().show()

+----------------+-----+
|  payment_method|count|
+----------------+-----+
|     Credit Card|25062|
|          M-Pesa|24759|
|   Bank Transfer|25043|
|Cash on Delivery|25136|
+----------------+-----+



In [68]:
#how many customers were 1st time clients
dataset.groupBy('is_first_purchase').count().show()
#true=1st time
#false=returning clients

+-----------------+-----+
|is_first_purchase|count|
+-----------------+-----+
|             true|49961|
|            false|50039|
+-----------------+-----+



In [41]:
#how many devices types were used by clients
dataset.select('device_type').distinct().show()

+-----------+
|device_type|
+-----------+
|     Mobile|
|     Tablet|
|    Desktop|
+-----------+



In [43]:
#how customers used tablets
dataset.groupBy('device_type').count().show()

+-----------+-----+
|device_type|count|
+-----------+-----+
|     Mobile|33343|
|     Tablet|33306|
|    Desktop|33351|
+-----------+-----+



In [53]:
#how many distinct users made purchases?
dataset.describe('user_id').show()

+-------+--------------------+
|summary|             user_id|
+-------+--------------------+
|  count|              100000|
|   mean|                NULL|
| stddev|                NULL|
|    min|0000314a-4616-4f2...|
|    max|ffffb4eb-f1c5-420...|
+-------+--------------------+



In [55]:
#how many transactions were from thika
dataset.groupBy('city').count().show()

+-------+-----+
|   city|count|
+-------+-----+
| Nakuru| 9927|
| Kisumu|10015|
|Mombasa|10012|
|Eldoret|10006|
|Garissa| 9983|
|Nairobi| 9999|
|  Thika|10081|
| Kitale| 9989|
|   Meru|10006|
|  Nyeri| 9982|
+-------+-----+



In [64]:
#alternative
dataset.filter(dataset.city=='Thika').count()

10081

In [69]:
#how many clients ordered 3 or more quantities of commodities
dataset.filter(dataset.quantity >=3).count()

50146